# LeetCode 745 — Prefix and Suffix Search bằng Python

- Link: https://leetcode.com/problems/prefix-and-suffix-search/
- Độ khó: **Hard**
- Chủ đề: Thiết kế lớp, Trie (cây tiền tố), Hash Map, Tiền xử lý để truy vấn nhanh, Giao hai danh sách đã sắp xếp

Skeleton trên LeetCode (Python3):

```python
class WordFilter:

    def __init__(self, words: List[str]):

    def f(self, pref: str, suff: str) -> int:


# Your WordFilter object will be instantiated and called as such:
# obj = WordFilter(words)
# param_1 = obj.f(pref,suff)
```

## 1. Giới thiệu & mục tiêu học

Ta có một từ điển và cần trả lời rất nhiều câu hỏi dạng: "**từ nào (có chỉ số lớn nhất) vừa bắt đầu bằng `pref` vừa kết thúc bằng `suff`?**". Một câu hỏi thì dễ; cái khó là **hàng chục nghìn câu hỏi**, nên phải **tiền xử lý** để mỗi câu trả lời thật nhanh.

Điểm hay của bài: các từ rất ngắn (≤ 7 ký tự). Điều đó cho phép những lời giải "liệt kê hết mọi tổ hợp" — một bài học quan trọng: **đọc kỹ ràng buộc để chọn chiến lược**.

Sau notebook này bạn sẽ:

1. Viết lời giải quét tuyến tính (baseline) và biết vì sao nó chậm.
2. Dùng **hash map của mọi cặp (tiền tố, hậu tố)** → truy vấn `O(1)`.
3. Hiểu và cài **Trie** từ đầu, rồi mẹo "**hậu tố + '#' + từ**" để một Trie trả lời được cả tiền tố lẫn hậu tố.
4. Dùng **hai Trie** (xuôi và ngược) và **giao hai danh sách chỉ số đã sắp xếp** từ cuối.
5. Cân nhắc đánh đổi bộ nhớ – thời gian giữa các cách.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Thiết kế lớp `WordFilter`:

- `WordFilter(words)`: khởi tạo với danh sách từ `words`.
- `f(pref, suff)`: trả về **chỉ số lớn nhất** `i` sao cho `words[i]` có tiền tố `pref` **và** hậu tố `suff`. Nếu không có, trả `-1`.

(Nếu một từ xuất hiện nhiều lần, chỉ số lớn nhất được tính.)

**Ví dụ**

```text
Input:
["WordFilter", "f"]
[[["apple"]], ["a", "e"]]
Output:
[null, 0]
Giải thích: words[0] = "apple" bắt đầu bằng "a" và kết thúc bằng "e".
```

**Ràng buộc**

- `1 <= words.length <= 10^4`
- `1 <= words[i].length <= 7`
- `1 <= pref.length, suff.length <= 7`
- Chỉ gồm chữ cái tiếng Anh thường.
- Gọi `f` tối đa `10^4` lần.

In [1]:
from typing import List

def run(cls, ops, args):
    out, obj = [], None
    for op, a in zip(ops, args):
        if op == "WordFilter":
            obj = cls(*a); out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

EX_OPS = ["WordFilter", "f"]
EX_ARGS = [[["apple"]], ["a", "e"]]
EX_OUT = [None, 0]

# Thêm một bộ ví dụ tự tạo, có từ trùng và nhiều đáp án
WORDS2 = ["apple", "ape", "apply", "maple", "apple", "ample"]
QUERIES2 = [("ap", "e"), ("a", "le"), ("m", "e"), ("ap", "ly"), ("b", "e"), ("ample", "ample"), ("apple", "e")]
EXPECT2 = [4, 5, 3, 2, -1, 5, 4]
print(list(zip(QUERIES2, EXPECT2)))

[(('ap', 'e'), 4), (('a', 'le'), 5), (('m', 'e'), 3), (('ap', 'ly'), 2), (('b', 'e'), -1), (('ample', 'ample'), 5), (('apple', 'e'), 4)]


## 3. Kiến thức nền

### 3.1. `str.startswith` và `str.endswith`

In [2]:
w = "apple"
print(w.startswith("ap"), w.endswith("le"), w.startswith("apples"), w.endswith(""))

True True False True


### 3.2. Liệt kê mọi tiền tố và hậu tố bằng slicing

Từ dài `L` có `L + 1` tiền tố (kể cả chuỗi rỗng và chính nó) và `L + 1` hậu tố. `w[:i]` là tiền tố dài `i`, `w[i:]` là hậu tố bắt đầu tại `i`.

In [3]:
w = "ape"
prefixes = [w[:i] for i in range(len(w) + 1)]
suffixes = [w[i:] for i in range(len(w) + 1)]
print(prefixes)
print(suffixes)

['', 'a', 'ap', 'ape']
['ape', 'pe', 'e', '']


### 3.3. `dict` với khoá là tuple

Tuple `(pref, suff)` là bất biến (immutable) nên làm khoá `dict` được. Gán lại cùng khoá sẽ **ghi đè** — tiện để giữ chỉ số lớn nhất nếu duyệt chỉ số tăng dần.

In [4]:
d = {}
for i, w in enumerate(["ab", "ab"]):
    d[("a", "b")] = i
print(d)

{('a', 'b'): 1}


### 3.4. Trie (cây tiền tố)

Trie là cây trong đó mỗi cạnh mang một ký tự; đường từ gốc tới một nút đọc ra một tiền tố. Ta cài nút bằng `dict` con: `node = {}`; `node.setdefault(ch, {})` đi xuống (tạo nếu chưa có). Thông tin phụ (ví dụ chỉ số lớn nhất) lưu vào một khoá đặc biệt.

In [5]:
def trie_insert(root, word, idx):
    node = root
    for ch in word:
        node = node.setdefault(ch, {})
        node["$"] = idx          # chỉ số lớn nhất đi qua nút này (vì idx tăng dần)

def trie_find(root, prefix):
    node = root
    for ch in prefix:
        if ch not in node:
            return -1
        node = node[ch]
    return node.get("$", -1)

root = {}
for i, w in enumerate(["apple", "ape", "bat"]):
    trie_insert(root, w, i)
print(trie_find(root, "ap"), trie_find(root, "b"), trie_find(root, "c"))

1 2 -1


### 3.5. Giao hai danh sách đã sắp xếp bằng hai con trỏ

Muốn tìm phần tử chung **lớn nhất** của hai list tăng dần: đặt hai con trỏ ở **cuối**, cái nào lớn hơn thì lùi cái đó; gặp bằng nhau thì đó là đáp án.

In [6]:
def max_common(a, b):
    i, j = len(a) - 1, len(b) - 1
    while i >= 0 and j >= 0:
        if a[i] == b[j]:
            return a[i]
        if a[i] > b[j]:
            i -= 1
        else:
            j -= 1
    return -1

print(max_common([0, 2, 5, 7], [1, 2, 5, 6]), max_common([1, 3], [2, 4]))

5 -1


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Có bao nhiêu cặp (tiền tố, hậu tố) cho một từ?

Từ dài `L ≤ 7` có `L` tiền tố khác rỗng và `L` hậu tố khác rỗng → tối đa `7 × 7 = 49` cặp. Với `10^4` từ: tối đa `4.9·10^5` cặp — **hoàn toàn lưu được** trong bộ nhớ!

### 4.2. Tính tay với `WORDS2 = ["apple", "ape", "apply", "maple", "apple", "ample"]`

Truy vấn `("ap", "e")`:

| i | từ | bắt đầu "ap"? | kết thúc "e"? | khớp |
|---|---|---|---|---|
| 5 | ample | không | có | ✗ |
| 4 | apple | có | có | ✔ → trả 4 |

Duyệt **từ cuối** giúp dừng ngay khi khớp đầu tiên.

### 4.3. Mẹo ghép chuỗi cho Trie

Với từ `"ape"`, xét các chuỗi `hậu_tố + "#" + từ`: `"ape#ape"`, `"pe#ape"`, `"e#ape"`, `"#ape"`. Một truy vấn `(pref, suff)` khớp từ `w` ⇔ chuỗi `suff + "#" + pref` là **tiền tố** của chuỗi `suff + "#" + w` (vì `suff` là hậu tố của `w`). Vậy chỉ cần **một Trie** chứa mọi chuỗi ghép.

In [7]:
w = "ape"
wrapped = [w[i:] + "#" + w for i in range(len(w) + 1)]
print(wrapped)
q = "e" + "#" + "ap"
print(q, "là tiền tố của", [x for x in wrapped if x.startswith(q)])

['ape#ape', 'pe#ape', 'e#ape', '#ape']
e#ap là tiền tố của ['e#ape']


## 5. Cách 1 — Brute Force: quét từ cuối

**Ý tưởng.** Với mỗi truy vấn, duyệt `i` từ `len(words)-1` về `0`, trả về `i` đầu tiên thoả `startswith(pref) and endswith(suff)`.

In [8]:
class WordFilterBrute:
    def __init__(self, words):
        self.words = words

    def f(self, pref, suff):
        for i in range(len(self.words) - 1, -1, -1):
            w = self.words[i]
            if w.startswith(pref) and w.endswith(suff):
                return i
        return -1

print(run(WordFilterBrute, EX_OPS, EX_ARGS))
wf = WordFilterBrute(WORDS2)
print([wf.f(p, s) for p, s in QUERIES2])

[None, 0]
[4, 5, 3, 2, -1, 5, 4]


**Độ phức tạp.** Khởi tạo `O(1)`; mỗi truy vấn `O(N · L)`. Tổng `Q · N · L = 10^4 · 10^4 · 7 = 7·10^8` — quá chậm trong Python khi không có từ khớp.

**Nút thắt.** Mỗi truy vấn duyệt lại toàn bộ từ điển. Vì số cặp (tiền tố, hậu tố) **nhỏ** (≤ 49 mỗi từ), ta có thể **tính sẵn mọi câu trả lời**.

## 6. Bước đột phá

**Insight 1 — liệt kê hết.** Mỗi từ chỉ sinh ≤ 49 cặp `(pref, suff)`. Lưu `ans[(pref, suff)] = i` cho mọi cặp, duyệt `i` tăng dần để giá trị cuối cùng là chỉ số **lớn nhất**. Truy vấn = tra `dict`.

**Insight 2 — Trie ghép chuỗi.** Thay vì lưu cặp, lưu chuỗi `suff + '#' + word` vào Trie; mỗi nút giữ chỉ số lớn nhất đi qua. Truy vấn = đi theo `suff + '#' + pref`.

**Insight 3 — hai Trie + giao.** Trie xuôi cho tiền tố → danh sách chỉ số có tiền tố đó (tăng dần); Trie ngược cho hậu tố → danh sách chỉ số. Đáp án = phần tử chung lớn nhất.

**Bất biến (cho Insight 1, 2).** Sau khi chèn các từ `0..i`, giá trị lưu cho mỗi khoá/nút là chỉ số **lớn nhất** trong `0..i` có khoá/nút đó — vì ta ghi đè theo thứ tự tăng dần.

**Quy tắc nếu… thì…**

- Nếu từ ngắn và số truy vấn lớn → tiền xử lý mọi cặp (Cách 2/3).
- Nếu từ dài (ví dụ 10^3 ký tự) → số cặp `L^2` bùng nổ → dùng hai Trie + giao (Cách 4) hoặc cấu trúc khác.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Hash map mọi cặp (tiền tố, hậu tố)

**Giải thích.** Với mỗi từ `w` (chỉ số `i` tăng dần), với mọi `a` từ 1..L và `b` từ 0..L-1: `ans[(w[:a], w[b:])] = i`.

In [9]:
class WordFilterHash:
    def __init__(self, words):
        self.ans = {}
        for i, w in enumerate(words):
            L = len(w)
            prefs = [w[:a] for a in range(1, L + 1)]
            suffs = [w[b:] for b in range(L)]
            for p in prefs:
                for s in suffs:
                    self.ans[(p, s)] = i

    def f(self, pref, suff):
        return self.ans.get((pref, suff), -1)

print(run(WordFilterHash, EX_OPS, EX_ARGS))
wf = WordFilterHash(WORDS2)
print([wf.f(p, s) for p, s in QUERIES2])

[None, 0]
[4, 5, 3, 2, -1, 5, 4]


**Mẹo nhỏ:** nếu có nhiều từ trùng nhau, có thể bỏ qua các lần xuất hiện trước của cùng một từ (chỉ chèn lần cuối) — giảm thời gian khởi tạo.

**Code nộp LeetCode (Cách 2).**

```python
class WordFilter:

    def __init__(self, words: List[str]):
        last = {w: i for i, w in enumerate(words)}   # chỉ số cuối của mỗi từ
        self.ans = {}
        for w, i in sorted(last.items(), key=lambda x: x[1]):
            L = len(w)
            suffs = [w[b:] for b in range(L)]
            for a in range(1, L + 1):
                p = w[:a]
                for s in suffs:
                    self.ans[(p, s)] = i

    def f(self, pref: str, suff: str) -> int:
        return self.ans.get((pref, suff), -1)
```

**Độ phức tạp.** Khởi tạo `O(N · L^2)` cặp (mỗi cặp băm chuỗi dài `O(L)` → `O(N·L^3)` thao tác ký tự, `L ≤ 7` nên nhỏ). Truy vấn `O(L)` (băm). Bộ nhớ `O(N · L^2)` khoá.

### 7.2. Cách 3 — Một Trie với chuỗi `suff + '#' + word`

**Giải thích.** Chèn mọi chuỗi `w[b:] + '#' + w` (với `b = 0..L`) vào Trie; tại mỗi nút đi qua ghi `idx = i`. Truy vấn: đi theo `suff + '#' + pref`; nếu đi được hết thì trả `idx` của nút cuối.

Để nhanh hơn, ta cài Trie bằng **mảng các dict** (`children[node]`) và mảng `best[node]` thay vì dict lồng có khoá đặc biệt.

In [10]:
class WordFilterTrie:
    def __init__(self, words):
        self.children = [{}]          # children[node][ch] = node con
        self.best = [-1]
        for i, w in enumerate(words):
            for b in range(len(w) + 1):
                node = 0
                for ch in w[b:] + "#" + w:
                    nxt = self.children[node].get(ch)
                    if nxt is None:
                        nxt = len(self.children)
                        self.children.append({})
                        self.best.append(-1)
                        self.children[node][ch] = nxt
                    node = nxt
                    self.best[node] = i

    def f(self, pref, suff):
        node = 0
        for ch in suff + "#" + pref:
            node = self.children[node].get(ch)
            if node is None:
                return -1
        return self.best[node]

print(run(WordFilterTrie, EX_OPS, EX_ARGS))
wf = WordFilterTrie(WORDS2)
print([wf.f(p, s) for p, s in QUERIES2], " số nút:", len(wf.children))

[None, 0]
[4, 5, 3, 2, -1, 5, 4]  số nút: 173


**Code nộp LeetCode (Cách 3).**

```python
class WordFilter:

    def __init__(self, words: List[str]):
        self.children = [{}]
        self.best = [-1]
        for i, w in enumerate(words):
            for b in range(len(w) + 1):
                node = 0
                for ch in w[b:] + '#' + w:
                    nxt = self.children[node].get(ch)
                    if nxt is None:
                        nxt = len(self.children)
                        self.children.append({})
                        self.best.append(-1)
                        self.children[node][ch] = nxt
                    node = nxt
                    self.best[node] = i

    def f(self, pref: str, suff: str) -> int:
        node = 0
        for ch in suff + '#' + pref:
            node = self.children[node].get(ch)
            if node is None:
                return -1
        return self.best[node]
```

**Độ phức tạp.** Khởi tạo: mỗi từ chèn `L + 1` chuỗi dài `≤ 2L + 1` → `O(N · L^2)`. Truy vấn `O(|pref| + |suff|)`. Bộ nhớ `O(N · L^2)` nút (trường hợp xấu).

### 7.3. Cách 4 — Hai Trie (xuôi, ngược) + giao danh sách chỉ số

**Giải thích.**

- Trie xuôi: chèn `w`; mỗi nút giữ list các chỉ số đi qua (tự động **tăng dần** vì chèn theo `i` tăng).
- Trie ngược: chèn `w[::-1]` tương tự.
- Truy vấn: lấy list `A` tại nút của `pref` (Trie xuôi), list `B` tại nút của `suff[::-1]` (Trie ngược), tìm phần tử chung lớn nhất bằng hai con trỏ từ cuối (mục 3.5).

Bộ nhớ chỉ `O(N · L)` (mỗi từ đóng góp `L` chỉ số vào mỗi Trie) — tốt khi từ dài. Đổi lại truy vấn có thể chậm `O(N)` trường hợp xấu (ví dụ `pref = "a"` và `suff = "z"` với rất nhiều từ).

In [11]:
class WordFilterTwoTries:
    def __init__(self, words):
        self.fw = {}
        self.bw = {}
        for i, w in enumerate(words):
            for root, s in ((self.fw, w), (self.bw, w[::-1])):
                node = root
                for ch in s:
                    node = node.setdefault(ch, {})
                    node.setdefault("$", []).append(i)

    @staticmethod
    def _walk(root, s):
        node = root
        for ch in s:
            node = node.get(ch)
            if node is None:
                return []
        return node.get("$", [])

    def f(self, pref, suff):
        A = self._walk(self.fw, pref)
        B = self._walk(self.bw, suff[::-1])
        return max_common(A, B)

print(run(WordFilterTwoTries, EX_OPS, EX_ARGS))
wf = WordFilterTwoTries(WORDS2)
print([wf.f(p, s) for p, s in QUERIES2])

[None, 0]
[4, 5, 3, 2, -1, 5, 4]


> Chú ý: một từ xuất hiện 2 lần sẽ có 2 chỉ số khác nhau trong list — không sao, ta chỉ cần phần tử chung lớn nhất.

**Code nộp LeetCode (Cách 4).**

```python
class WordFilter:

    def __init__(self, words: List[str]):
        self.fw, self.bw = {}, {}
        for i, w in enumerate(words):
            for root, s in ((self.fw, w), (self.bw, w[::-1])):
                node = root
                for ch in s:
                    node = node.setdefault(ch, {})
                    node.setdefault('$', []).append(i)

    def _walk(self, root, s):
        node = root
        for ch in s:
            node = node.get(ch)
            if node is None:
                return []
        return node.get('$', [])

    def f(self, pref: str, suff: str) -> int:
        A = self._walk(self.fw, pref)
        B = self._walk(self.bw, suff[::-1])
        i, j = len(A) - 1, len(B) - 1
        while i >= 0 and j >= 0:
            if A[i] == B[j]:
                return A[i]
            if A[i] > B[j]:
                i -= 1
            else:
                j -= 1
        return -1
```

**Độ phức tạp.** Khởi tạo `O(N · L)`; truy vấn `O(L + |A| + |B|)` — xấu nhất `O(N)`. Bộ nhớ `O(N · L)`.

## 8. TRACE từng bước

### 8.1. Trace Cách 3: chèn `"ape"` (i = 1) và truy vấn

Các chuỗi chèn: `"ape#ape"`, `"pe#ape"`, `"e#ape"`, `"#ape"`. Truy vấn `f("ap", "e")` → đi theo `"e#ap"`: `e → # → a → p`, tới nút có `best = 1`.

In [12]:
def trace_trie(words, queries):
    children, best = [{}], [-1]
    for i, w in enumerate(words):
        for b in range(len(w) + 1):
            s = w[b:] + "#" + w
            node, created = 0, 0
            for ch in s:
                nxt = children[node].get(ch)
                if nxt is None:
                    nxt = len(children); children.append({}); best.append(-1)
                    children[node][ch] = nxt; created += 1
                node = nxt; best[node] = i
            print(f"chèn {s!r:<12} (i={i}): tạo {created} nút mới, tổng {len(children)} nút")
    for p, s in queries:
        q = s + "#" + p
        node, path = 0, []
        for ch in q:
            node = children[node].get(ch)
            if node is None:
                path.append(f"{ch}:không có"); break
            path.append(f"{ch}->nút{node}(best={best[node]})")
        ans = best[node] if node is not None else -1
        print(f"f({p!r},{s!r}) đi theo {q!r}: {' '.join(path)} => {ans}")

trace_trie(["apple", "ape"], [("ap", "e"), ("app", "le"), ("ap", "x")])

chèn 'apple#apple' (i=0): tạo 11 nút mới, tổng 12 nút
chèn 'pple#apple' (i=0): tạo 10 nút mới, tổng 22 nút
chèn 'ple#apple'  (i=0): tạo 8 nút mới, tổng 30 nút
chèn 'le#apple'   (i=0): tạo 8 nút mới, tổng 38 nút
chèn 'e#apple'    (i=0): tạo 7 nút mới, tổng 45 nút
chèn '#apple'     (i=0): tạo 6 nút mới, tổng 51 nút
chèn 'ape#ape'    (i=1): tạo 5 nút mới, tổng 56 nút
chèn 'pe#ape'     (i=1): tạo 5 nút mới, tổng 61 nút
chèn 'e#ape'      (i=1): tạo 1 nút mới, tổng 62 nút
chèn '#ape'       (i=1): tạo 1 nút mới, tổng 63 nút
f('ap','e') đi theo 'e#ap': e->nút38(best=1) #->nút39(best=1) a->nút40(best=1) p->nút41(best=1) => 1
f('app','le') đi theo 'le#app': l->nút30(best=0) e->nút31(best=0) #->nút32(best=0) a->nút33(best=0) p->nút34(best=0) p->nút35(best=0) => 0
f('ap','x') đi theo 'x#ap': x:không có => -1


### 8.2. Trace Cách 4: giao hai danh sách

In [13]:
def trace_two_tries(words, pref, suff):
    wf = WordFilterTwoTries(words)
    A = wf._walk(wf.fw, pref); B = wf._walk(wf.bw, suff[::-1])
    print(f"A (tiền tố {pref!r}) = {A}")
    print(f"B (hậu tố {suff!r}) = {B}")
    i, j = len(A) - 1, len(B) - 1
    while i >= 0 and j >= 0:
        print(f"  so A[{i}]={A[i]} với B[{j}]={B[j]}", end=" ")
        if A[i] == B[j]:
            print("-> bằng nhau, đáp án"); return A[i]
        if A[i] > B[j]:
            print("-> A lớn hơn, lùi i"); i -= 1
        else:
            print("-> B lớn hơn, lùi j"); j -= 1
    print("  hết -> -1"); return -1

print("Kết quả:", trace_two_tries(WORDS2, "ap", "e"))
print("Kết quả:", trace_two_tries(WORDS2, "a", "y"))

A (tiền tố 'ap') = [0, 1, 2, 4]
B (hậu tố 'e') = [0, 1, 3, 4, 5]
  so A[3]=4 với B[4]=5 -> B lớn hơn, lùi j
  so A[3]=4 với B[3]=4 -> bằng nhau, đáp án
Kết quả: 4
A (tiền tố 'a') = [0, 1, 2, 4, 5]
B (hậu tố 'y') = [2]
  so A[4]=5 với B[0]=2 -> A lớn hơn, lùi i
  so A[3]=4 với B[0]=2 -> A lớn hơn, lùi i
  so A[2]=2 với B[0]=2 -> bằng nhau, đáp án
Kết quả: 2


## 9. Code nộp LeetCode

Bản chính: **Cách 3 (một Trie ghép chuỗi)** — truy vấn `O(L)`, không phụ thuộc số từ khớp.

In [14]:
class WordFilter:

    def __init__(self, words: List[str]):
        self.children = [{}]
        self.best = [-1]
        for i, w in enumerate(words):
            for b in range(len(w) + 1):
                node = 0
                for ch in w[b:] + '#' + w:
                    nxt = self.children[node].get(ch)
                    if nxt is None:
                        nxt = len(self.children)
                        self.children.append({})
                        self.best.append(-1)
                        self.children[node][ch] = nxt
                    node = nxt
                    self.best[node] = i

    def f(self, pref: str, suff: str) -> int:
        node = 0
        for ch in suff + '#' + pref:
            node = self.children[node].get(ch)
            if node is None:
                return -1
        return self.best[node]

assert run(WordFilter, EX_OPS, EX_ARGS) == EX_OUT
wf = WordFilter(WORDS2)
assert [wf.f(p, s) for p, s in QUERIES2] == EXPECT2
print("WordFilter OK")

WordFilter OK


## 10. Vì sao thuật toán đúng

**Cách 2.** Mọi cặp hợp lệ `(pref, suff)` của truy vấn có `1 ≤ |pref|, |suff| ≤ 7`. Từ `w` khớp ⇔ `pref = w[:a]` với `a = |pref| ≤ L` và `suff = w[b:]` với `b = L - |suff| ≥ 0` ⇔ cặp `(pref, suff)` nằm trong tập ta đã chèn cho `w`. Do duyệt `i` tăng dần và ghi đè, giá trị lưu là chỉ số lớn nhất của từ khớp. Không có từ nào khớp ⇔ khoá không tồn tại → `-1`.

**Cách 3.** Bổ đề: `w` khớp `(pref, suff)` ⇔ `suff + '#' + pref` là tiền tố của chuỗi `w[b:] + '#' + w` với `b = L - |suff|`.

- (⇒) `w[b:] = suff`, và `pref` là tiền tố của `w`.
- (⇐) Ký tự `'#'` không xuất hiện trong từ, nên phần trước `'#'` của chuỗi chèn **phải đúng bằng** `suff` → `suff` là hậu tố của `w`; phần sau `'#'` bắt đầu bằng `pref` → `pref` là tiền tố của `w`.

Mỗi nút trên Trie ứng với một tiền tố của các chuỗi chèn; `best[node]` là chỉ số lớn nhất trong các từ có chuỗi chèn đi qua nút (ghi đè tăng dần). → đúng.

**Cách 4.** `A` = tập chỉ số có tiền tố `pref`, `B` = tập chỉ số có hậu tố `suff`; đáp án là `max(A ∩ B)`. Hai con trỏ từ cuối: nếu `A[i] > B[j]` thì `A[i]` không thể nằm trong `B` (mọi phần tử còn lại của `B` ≤ `B[j]` < `A[i]`) → bỏ `A[i]` an toàn; tương tự với `B[j]`.

## 11. Phân tích Time/Space

`N` = số từ, `L` = độ dài tối đa (7), `Q` = số truy vấn.

| Cách | Khởi tạo | Truy vấn | Bộ nhớ |
|---|---|---|---|
| 1. Quét | `O(1)` | `O(N·L)` | `O(1)` thêm |
| 2. Hash cặp | `O(N·L^3)` thao tác ký tự | `O(L)` | `O(N·L^2)` khoá |
| 3. Trie ghép | `O(N·L^2)` | `O(L)` | `O(N·L^2)` nút |
| 4. Hai Trie | `O(N·L)` | `O(L + N)` xấu nhất | `O(N·L)` |

Thử với `N = 10^4`, `Q = 10^4` (đúng giới hạn đề):

In [15]:
import random, string, time
rng = random.Random(745)
words = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 7))) for _ in range(10000)]
queries = [("".join(rng.choice("abc") for _ in range(rng.randint(1, 3))),
            "".join(rng.choice("abcd") for _ in range(rng.randint(1, 3)))) for _ in range(10000)]
for cls in [WordFilterBrute, WordFilterHash, WordFilterTrie, WordFilterTwoTries]:
    t0 = time.perf_counter(); obj = cls(words); t1 = time.perf_counter()
    qs = queries if cls is not WordFilterBrute else queries[:500]
    res = [obj.f(p, s) for p, s in qs]; t2 = time.perf_counter()
    print(f"{cls.__name__:<20} init {1000*(t1-t0):7.1f} ms | {len(qs):>5} truy vấn {1000*(t2-t1):7.1f} ms")

WordFilterBrute      init     0.0 ms |   500 truy vấn   109.3 ms


WordFilterHash       init    34.8 ms | 10000 truy vấn     1.6 ms


WordFilterTrie       init    52.5 ms | 10000 truy vấn     2.9 ms
WordFilterTwoTries   init    11.3 ms | 10000 truy vấn    17.3 ms


## 12. Bộ test

In [16]:
ALL = [WordFilterBrute, WordFilterHash, WordFilterTrie, WordFilterTwoTries, WordFilter]
for cls in ALL:
    assert run(cls, EX_OPS, EX_ARGS) == EX_OUT
    obj = cls(WORDS2)
    assert [obj.f(p, s) for p, s in QUERIES2] == EXPECT2, cls.__name__

EDGE_WORDS = ["a", "b", "ab", "ba", "a"]
EDGE_Q = [("a", "a", 4), ("b", "a", 3), ("a", "b", 2), ("ab", "ab", 2), ("aa", "a", -1),
          ("a", "ba", -1), ("ba", "a", 3), ("abc", "c", -1)]
for cls in ALL:
    obj = cls(EDGE_WORDS)
    for p, s, e in EDGE_Q:
        assert obj.f(p, s) == e, (cls.__name__, p, s, obj.f(p, s), e)
print("Ví dụ + edge case OK")

Ví dụ + edge case OK


In [17]:
rng = random.Random(7450)
for _ in range(300):
    ws = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(1, 25))]
    objs = [cls(ws) for cls in ALL]
    for _ in range(30):
        p = "".join(rng.choice("ab") for _ in range(rng.randint(1, 4)))
        s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 4)))
        res = [o.f(p, s) for o in objs]
        assert len(set(res)) == 1, (ws, p, s, res)
print("300 từ điển ngẫu nhiên × 30 truy vấn: mọi cách khớp brute force")

300 từ điển ngẫu nhiên × 30 truy vấn: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Giữ chỉ số **nhỏ nhất** thay vì lớn nhất

**Code sai:** dùng `setdefault` (giữ giá trị đầu tiên) hoặc duyệt từ cuối rồi ghi đè.

**Code đúng:** duyệt `i` tăng dần và **ghi đè**.

In [18]:
class HashFirstWrong(WordFilterHash):
    def __init__(self, words):
        self.ans = {}
        for i, w in enumerate(words):
            for a in range(1, len(w) + 1):
                for b in range(len(w)):
                    self.ans.setdefault((w[:a], w[b:]), i)      # SAI: giữ chỉ số đầu

print("sai:", HashFirstWrong(WORDS2).f("ap", "e"), " đúng:", WordFilter(WORDS2).f("ap", "e"))

sai: 0  đúng: 4


### Lỗi 2 — Ghép `pref + '#' + suff` sai thứ tự / quên ký tự phân cách

**Code sai:** chèn `w[b:] + w` (không có `'#'`) → truy vấn `suff + pref` có thể khớp nhầm khi ranh giới bị lệch.

**Ví dụ:** từ `"ab"` sinh chuỗi chèn `"abab"`. Truy vấn `f("b", "a")` đi theo `"a" + "b" = "ab"` — là tiền tố của `"abab"` nên trả về chỉ số của `"ab"`, dù `"ab"` không bắt đầu bằng `"b"`. Cell dưới để máy tự tìm phản ví dụ.

**Code đúng:** luôn có ký tự phân cách không thuộc bảng chữ cái.

In [19]:
class TrieNoSepWrong:
    def __init__(self, words):
        self.root = {}
        for i, w in enumerate(words):
            for b in range(len(w) + 1):
                node = self.root
                for ch in w[b:] + w:                            # SAI: thiếu '#'
                    node = node.setdefault(ch, {}); node["$"] = i
    def f(self, pref, suff):
        node = self.root
        for ch in suff + pref:
            if ch not in node: return -1
            node = node[ch]
        return node["$"]

rng2 = random.Random(2)
for _ in range(2000):
    ws = ["".join(rng2.choice("ab") for _ in range(rng2.randint(1, 3))) for _ in range(3)]
    p = "".join(rng2.choice("ab") for _ in range(rng2.randint(1, 2)))
    s = "".join(rng2.choice("ab") for _ in range(rng2.randint(1, 2)))
    if TrieNoSepWrong(ws).f(p, s) != WordFilterBrute(ws).f(p, s):
        print("Phản ví dụ:", ws, (p, s), "sai:", TrieNoSepWrong(ws).f(p, s), "đúng:", WordFilterBrute(ws).f(p, s))
        break

Phản ví dụ: ['ab', 'a', 'baa'] ('ab', 'ba') sai: 2 đúng: -1


### Lỗi 3 — Hai Trie nhưng quên đảo `suff`

**Code sai:** tra Trie ngược bằng `suff` thay vì `suff[::-1]`.

**Code đúng:** Trie ngược chứa `w[::-1]`, nên hậu tố `suff` của `w` là tiền tố `suff[::-1]` của `w[::-1]`.

In [20]:
wf4 = WordFilterTwoTries(["apple"])
print("sai:", max_common(wf4._walk(wf4.fw, "a"), wf4._walk(wf4.bw, "le")),
      " đúng:", wf4.f("a", "le"))

sai: -1  đúng: 0


### Lỗi 4 — Ước lượng sai bộ nhớ khi từ dài

**Code sai (tư duy):** áp dụng Cách 2/3 cho từ dài `L = 1000` → `L^2 = 10^6` cặp mỗi từ.

**Code đúng:** đọc ràng buộc! `L ≤ 7` mới cho phép liệt kê. Với từ dài, dùng Cách 4.

## 14. Mẹo & nhận dạng pattern

- **Ràng buộc nhỏ bất thường** (`L ≤ 7`, `n ≤ 20`) → nghĩ tới **liệt kê toàn bộ** rồi tra cứu.
- **Truy vấn tiền tố** → Trie. **Truy vấn hậu tố** → Trie trên chuỗi đảo.
- **Truy vấn kết hợp hai điều kiện chuỗi** → ghép bằng **ký tự phân cách** (`'#'`, `'{'`) để một Trie xử lý.
- **"Chỉ số lớn nhất thoả mãn"** → chèn theo thứ tự tăng và ghi đè; hoặc giữ list tăng dần và duyệt từ cuối.
- Bài liên quan: *Implement Trie (208)*, *Design Add and Search Words (211)*, *Search Suggestions System (1268)*, *Stream of Characters (1032)*.

## 15. Biến thể

1. Trả về **tất cả** chỉ số khớp (hoặc số lượng): Cách 4 giao toàn bộ hai list.
2. Thêm điều kiện **chứa chuỗi con** `mid`: lọc thêm sau khi giao.
3. Hỗ trợ **thêm từ động** (`addWord`): Cách 3 chèn tiếp được ngay.
4. Truy vấn **chỉ tiền tố** hoặc **chỉ hậu tố** (một trong hai rỗng): Cách 3 xử lý được nếu chèn cả `b = L` (chuỗi `'#' + w`).
5. Ký tự wildcard `.` trong `pref`: DFS trên Trie như bài 211.

In [21]:
# Biến thể 1: đếm số từ khớp (pref, suff) bằng hai Trie
def count_matches(wf, pref, suff):
    A = set(wf._walk(wf.fw, pref)); B = wf._walk(wf.bw, suff[::-1])
    return sum(1 for x in B if x in A)

wf = WordFilterTwoTries(WORDS2)
print(count_matches(wf, "a", "e"), sum(w.startswith("a") and w.endswith("e") for w in WORDS2))

# Biến thể 4: truy vấn chỉ hậu tố với Cách 3 (pref rỗng)
wf3 = WordFilter(WORDS2)
print("chỉ hậu tố 'le':", wf3.f("", "le"), " chỉ tiền tố 'ma':", wf3.f("ma", ""))

4 4
chỉ hậu tố 'le': 5  chỉ tiền tố 'ma': 3


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Init | Truy vấn | Bộ nhớ | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Quét | `startswith`/`endswith` từ cuối | `O(1)` | `O(N·L)` | nhỏ | Ít truy vấn, trọng tài |
| 2. Hash cặp | Lưu mọi `(pref, suff)` | `O(N·L^2)` | `O(L)` | `O(N·L^2)` | `L` nhỏ, code ngắn nhất |
| 3. Trie ghép | `suff#word` | `O(N·L^2)` | `O(L)` | `O(N·L^2)` | `L` nhỏ, chuẩn phỏng vấn |
| 4. Hai Trie | Giao hai list chỉ số | `O(N·L)` | `O(L+N)` | `O(N·L)` | Từ dài, bộ nhớ hạn chế |

## 17. Bài tập

**Bài 1 (đoán output).** `WordFilter(["cabaabaaaa","ccbcababac","bacaabccba","bcbbcbacaa","abcaccbcaa","accabaccaa","cabcbbbcca","ababccabcb","caccbbcbab","bccbacbcba"])`. Gọi `f("bccbacbcba","a")`, `f("ab","abcaccbcaa")`, `f("a","aa")`, `f("cabaaba","abaaaa")`. Kết quả?

**Bài 2 (tính tay).** Liệt kê mọi chuỗi chèn vào Trie (Cách 3) cho từ `"cat"`. Có bao nhiêu chuỗi?

**Bài 3 (điền TODO).** Hoàn thiện `all_pairs(w)` trả về tập mọi cặp `(pref, suff)` khác rỗng của `w`.

**Bài 4 (tự viết lại).** Viết lại Cách 3 với Trie dạng dict lồng (khoá `"$"` lưu chỉ số).

**Bài 5 (mini challenge).** Thêm `add_word(w)` vào Cách 3 để hỗ trợ thêm từ sau khi khởi tạo (chỉ số mới = số từ hiện có).

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def all_pairs(w):
    # TODO: {(w[:a], w[b:]) for a in 1..L for b in 0..L-1}
    pass

# Bài 4 — TODO
class MyWordFilter:
    def __init__(self, words):
        # TODO: self.root = {}; chèn w[b:] + '#' + w, ghi node['$'] = i
        pass
    def f(self, pref, suff):
        # TODO: đi theo suff + '#' + pref
        pass

# Bài 5 — TODO
class DynamicWordFilter:
    def __init__(self, words):
        # TODO: self.count = 0; gọi add_word cho từng từ
        pass
    def add_word(self, w):
        # TODO: chèn với chỉ số self.count; self.count += 1
        pass
    def f(self, pref, suff):
        pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: kiểm tra từng truy vấn bằng `startswith`/`endswith`, duyệt từ cuối.
- Mức 2: `"bccbacbcba"` là từ cuối (chỉ số 9) và kết thúc bằng `a` → 9.
- Mức 3: kết quả `[9, 4, 5, 0]`.

**Bài 2.**
- Mức 1: hậu tố của `cat`: `cat`, `at`, `t`, `""`.
- Mức 2: ghép với `'#cat'`.
- Mức 3: `cat#cat`, `at#cat`, `t#cat`, `#cat` → 4 chuỗi (`L + 1`).

**Bài 3.**
- Mức 1: hai vòng lặp lồng.
- Mức 2: tiền tố khác rỗng: `a` từ 1; hậu tố khác rỗng: `b` tới `L-1`.
- Mức 3: dùng set comprehension.

**Bài 4.**
- Mức 1: `node = node.setdefault(ch, {})`.
- Mức 2: `node['$'] = i` tại mọi nút đi qua.
- Mức 3: truy vấn trả `node.get('$', -1)`; cẩn thận đừng đi vào khoá `'$'` như một ký tự (ký tự trong từ không bao giờ là `'$'`).

**Bài 5.**
- Mức 1: tách vòng chèn thành hàm riêng.
- Mức 2: chỉ số tăng dần nên ghi đè vẫn giữ max.
- Mức 3: kiểm tra bằng brute force sau mỗi lần thêm.

## 19. Đáp án tham khảo

In [23]:
# Bài 1
W1 = ["cabaabaaaa","ccbcababac","bacaabccba","bcbbcbacaa","abcaccbcaa",
      "accabaccaa","cabcbbbcca","ababccabcb","caccbbcbab","bccbacbcba"]
wf = WordFilter(W1)
res = [wf.f("bccbacbcba", "a"), wf.f("ab", "abcaccbcaa"), wf.f("a", "aa"), wf.f("cabaaba", "abaaaa")]
assert res == [9, 4, 5, 0], res

# Bài 2
assert ["cat"[b:] + "#cat" for b in range(4)] == ["cat#cat", "at#cat", "t#cat", "#cat"]

# Bài 3
def all_pairs_ans(w):
    L = len(w)
    return {(w[:a], w[b:]) for a in range(1, L + 1) for b in range(L)}

assert len(all_pairs_ans("abc")) == 9 and ("ab", "bc") in all_pairs_ans("abc")
print("Bài 1–3 OK")

Bài 1–3 OK


In [24]:
# Bài 4
class MyWordFilterAns:
    def __init__(self, words):
        self.root = {}
        for i, w in enumerate(words):
            for b in range(len(w) + 1):
                node = self.root
                for ch in w[b:] + "#" + w:
                    node = node.setdefault(ch, {})
                    node["$"] = i
    def f(self, pref, suff):
        node = self.root
        for ch in suff + "#" + pref:
            if ch not in node:
                return -1
            node = node[ch]
        return node.get("$", -1)

# Bài 5
class DynamicWordFilterAns:
    def __init__(self, words):
        self.children, self.best, self.count = [{}], [-1], 0
        for w in words:
            self.add_word(w)
    def add_word(self, w):
        i = self.count; self.count += 1
        for b in range(len(w) + 1):
            node = 0
            for ch in w[b:] + "#" + w:
                nxt = self.children[node].get(ch)
                if nxt is None:
                    nxt = len(self.children); self.children.append({}); self.best.append(-1)
                    self.children[node][ch] = nxt
                node = nxt; self.best[node] = i
    def f(self, pref, suff):
        node = 0
        for ch in suff + "#" + pref:
            node = self.children[node].get(ch)
            if node is None:
                return -1
        return self.best[node]

rng = random.Random(11)
for _ in range(200):
    ws = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 15))]
    a = MyWordFilterAns(ws); d = DynamicWordFilterAns(ws[:len(ws) // 2])
    for w in ws[len(ws) // 2:]:
        d.add_word(w)
    b = WordFilterBrute(ws)
    for _ in range(20):
        p = "".join(rng.choice("abc") for _ in range(rng.randint(1, 3)))
        s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 3)))
        assert a.f(p, s) == b.f(p, s) == d.f(p, s)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template Trie ghép chuỗi:**

```python
for i, w in enumerate(words):          # i tăng dần -> ghi đè = giữ max
    for b in range(len(w) + 1):
        node = root
        for ch in w[b:] + '#' + w:
            node = child(node, ch); best[node] = i
# truy vấn: đi theo suff + '#' + pref, trả best[node] hoặc -1
```

**Template hash cặp:** `ans[(w[:a], w[b:])] = i` cho mọi `a ∈ [1, L]`, `b ∈ [0, L-1]`.

**Checklist trước khi submit:**

- [ ] Trả **chỉ số lớn nhất** (ghi đè theo `i` tăng dần).
- [ ] Có ký tự phân cách `'#'` giữa hậu tố và từ.
- [ ] Truy vấn đi theo `suff + '#' + pref` (thứ tự đúng!).
- [ ] Không tìm thấy → `-1`.
- [ ] Hai Trie: nhớ đảo `suff`.
- [ ] Ước lượng bộ nhớ: `N · L^2` với `L ≤ 7` là ổn.
- [ ] Tên lớp/method đúng: `WordFilter`, `f(pref, suff)`.